# 06 - Fluxo de decisao automatizado (LangGraph)

Demonstra o grafo completo (`src/assistant/graph.py`):

```
receber dados -> consultar prontuario -> checar exames pendentes
  -> rodar modelo de predicao de AVC -> sugerir conduta (LLM + RAG)
  -> aplicar guardrails -> [condicional] emitir alerta -> log de auditoria
```

Rodamos 3 cenarios: paciente de alto risco (deve alertar), paciente de baixo risco com exame pendente (nao deve alertar) e um caso onde o guardrail de prescricao direta e acionado (deve alertar mesmo com risco baixo).

In [ ]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ROOT = Path('/content/drive/MyDrive/stroke-prediction')
    sys.path.insert(0, str(ROOT))
    print("Rodando no Google Colab")
else:
    ROOT = Path('..').resolve()
    sys.path.insert(0, str(ROOT))
    print("Rodando Localmente")

from src.assistant.graph import run_flow
from src.assistant.llm_backend import get_generate_fn
from src.assistant.patient_db import build_patient_db
from src.assistant.retriever import build_vectorstore
from src.security.audit_log import read_audit_log

build_patient_db()
vectorstore = build_vectorstore()
generate_fn = get_generate_fn()

## Cenario 1 — paciente de alto risco (id 9046)

In [ ]:
result_high_risk = run_flow(
    patient_id=9046,
    question='Quais os proximos passos para este paciente?',
    vectorstore=vectorstore,
    generate_fn=generate_fn,
)
print('Risco de AVC:', result_high_risk['stroke_risk'])
print('Alerta emitido?', result_high_risk.get('alert'), '-', result_high_risk.get('alert_reason'))
print('Resposta:', result_high_risk['response'][:300])

## Cenario 2 — paciente de baixo risco com exame pendente (id 51676)

In [ ]:
result_low_risk = run_flow(
    patient_id=51676,
    question='Este paciente precisa de trombolise agora?',
    vectorstore=vectorstore,
    generate_fn=generate_fn,
)
print('Risco de AVC:', result_low_risk['stroke_risk'])
print('Exames pendentes:', result_low_risk['pending_exams'])
print('Alerta emitido?', result_low_risk.get('alert'))

## Cenario 3 — guardrail aciona alerta mesmo com risco baixo

In [ ]:
def unsafe_generate(prompt: str) -> str:
    # Simula uma saida de LLM que violaria a politica de nunca prescrever direto
    return 'Administre 10mg de enalapril e tome 500mg de AAS agora.'

result_guardrail = run_flow(
    patient_id=51676,
    question='O que fazer agora?',
    vectorstore=vectorstore,
    generate_fn=unsafe_generate,
)
print('Requer validacao humana?', result_guardrail['requires_human_validation'])
print('Alerta emitido?', result_guardrail.get('alert'), '-', result_guardrail.get('alert_reason'))
print('Resposta (com disclaimer):', result_guardrail['response'][-250:])

## Log de auditoria

Cada execucao do fluxo grava uma entrada em `results/audit_log.jsonl`.

In [ ]:
entries = read_audit_log()
print(f'{len(entries)} entradas no log de auditoria')
for e in entries[-3:]:
    print('-', e['timestamp'], '| paciente (pseudonimizado):', e['patient_id'], '| alerta de validacao:', e['requires_human_validation'])